##### **Gold Layer**
- ##### _Build dimensional model for analytics_

In [18]:
-- Create gold schema

CREATE SCHEMA IF NOT EXISTS gold;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [19]:
-- Validate silver tables

SHOW TABLES IN silver;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 20, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 3 fields>

In [20]:
-- Inspect customer structure before transformation

DESCRIBE silver.customers;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 21, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 3 fields>

In [21]:
-- Create Region Dimension

CREATE OR REPLACE TABLE gold.dim_region AS
SELECT DISTINCT
    CAST(RegionID AS STRING) AS RegionID,
    RegionName               AS RegionName
FROM silver.region;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 22, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [22]:
-- Create Geography Dimension

CREATE OR REPLACE TABLE gold.dim_geography AS
SELECT DISTINCT
    CAST(g.GeographyID AS STRING) AS GeographyKey,
    g.CountryName,
    g.StateCode,
    g.StateName,
    g.CityName,
    g.PostalCode,
    g.Latitude,
    g.Longitude,
    CAST(g.RegionID AS STRING) AS RegionID
FROM silver.geography g;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [23]:
-- Create Customer Dimension

CREATE OR REPLACE TABLE gold.dim_customer AS
SELECT
    CAST(REGEXP_REPLACE(c.CustomerID, '[^0-9]', '') AS BIGINT) AS CustomerKey,
    c.CustomerID,
    c.CustomerName,
    c.Gender,
    CAST(c.Age AS BIGINT) AS Age,
    c.AgeBand,
    c.SignupDate,
    c.CustomerType,
    c.LoyaltyTier,
    c.AcquisitionSource,
    ca.StateCode   AS HomeStateCode,
    ca.StateName   AS HomeState,
    ca.RegionName  AS HomeRegion,
    cs.CustomerSegmentName AS CustomerSegment
FROM silver.customers c
LEFT JOIN silver.customer_segments cs
    ON c.CustomerID = cs.CustomerID
LEFT JOIN silver.customer_addresses ca
    ON c.CustomerID = ca.CustomerID;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [24]:
-- Create Product Dimension

CREATE OR REPLACE TABLE gold.dim_product AS
SELECT
    CAST(REGEXP_REPLACE(p.ProductID, '[^0-9]', '') AS BIGINT) AS ProductKey,
    p.ProductID,
    p.SKU,
    p.ProductName,
    p.Brand,
    CAST(NULL AS BIGINT) AS CategoryKey,
    p.CategoryName,
    p.SubcategoryName,
    p.Department,
    p.ProductType,
    p.Color,
    p.Size,
    p.Season,
    CAST(p.UnitCostUSD AS DOUBLE) AS UnitCostUSD,
    CAST(p.StandardPriceUSD AS DOUBLE) AS StandardPriceUSD,
    CAST(p.IsActive AS BIGINT) AS IsActive
FROM silver.products p;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 25, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [25]:
-- Create Sales Fact Table

CREATE OR REPLACE TABLE gold.fact_sales_order_item AS
SELECT
    oi.OrderID,
    oi.OrderItemID,
    TO_DATE(CAST(oh.OrderDateKey AS STRING), 'yyyyMMdd') AS OrderDate,
    oh.CustomerKey,
    oi.ProductKey,
    ch.ChannelName AS CustomerChannel,
    oh.GeographyKey,
    os.OrderStatusName AS OrderStatus,
    oi.Quantity,
    oi.UnitPriceUSD,
    oi.GrossSalesUSD,
    oi.DiscountUSD,
    oi.ReturnAmountUSD,
    oi.NetRevenueUSD,
    oi.ProductCostUSD,
    oi.GrossProfitUSD,
    oi.IsReturned,
    oi.IsCancelled,
    oi.IsDelivered,
    oi.IsRepeatCustomer,
    oi.CustomerOrderSequence
FROM silver.order_items oi
LEFT JOIN silver.orders oh
    ON oi.OrderID = oh.OrderID
LEFT JOIN silver.order_statuses os
    ON oh.OrderStatusKey = os.OrderStatusID
LEFT JOIN silver.channels ch
    ON oh.CustomerChannelKey = ch.ChannelID;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 26, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [26]:
-- # Validate fact table

SELECT * FROM gold.fact_sales_order_item;

StatementMeta(, c997fd20-6cb8-4a19-97b1-650aa3f21b1a, 27, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 21 fields>